In [1]:
import os

os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/parastoof/data-science-project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="parastoof"
os.environ["MLFLOW_TRACKING_PASSWORD"]="a55b8c0ae54a9cdd2a268cee2b91031a09e6e24a"


In [2]:
import os
%pwd

'd:\\codes\\mlflow\\data-science-project\\reserch'

In [3]:
os.chdir("../")
%pwd

'd:\\codes\\mlflow\\data-science-project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: str
    target_column: str
    mlflow_url: str


In [6]:
from src.datascienceproject.constants import *
from src.datascienceproject.utils.common import read_yaml, create_directories, save_json

class ConfiguationManager:
    def __init__(self, 
                 config_pathfile=CONFIG_FILE_PATH,
                 params_pathfile=PARAMS_FILE_PATH,
                 schema_pathfile=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_pathfile)
        self.params=read_yaml(params_pathfile)
        self.schema=read_yaml(schema_pathfile)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)->ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        create_directories([config.root_dir])

        model_evaluation_config=ModelEvaluationConfig(root_dir=config.root_dir,
                                                    test_data_path=config.test_data_path,
                                                    model_path=config.model_path,
                                                    all_params=params,
                                                    metric_file_name=config.metric_file_name,
                                                    target_column=schema.name,
                                                    mlflow_url="https://dagshub.com/parastoof/data-science-project.mlflow"
                                                    )
        return model_evaluation_config

In [8]:
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from urllib.parse import urlparse


class ModelEvaluation:
    def __init__(self,config:ModelEvaluationConfig):
            self.config=config

    def eval_metrics(self, actual, pred):
          rmse=np.sqrt(mean_squared_error(actual, pred))
          mae=mean_absolute_error(actual, pred)
          r2=r2_score(actual, pred)
          return rmse, mae, r2

    def log_into_mlflow(self):
        test_data=pd.read_csv(self.config.test_data_path)
        model=joblib.load(self.config.model_path)
        test_x =test_data.drop([self.config.target_column], axis=1)
        test_y =test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_url)
        tracking_url_type_store=urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_quaities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_quaities)

            scores={"rmse":rmse, "mae":mae, "r2":r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)
            
            mlflow.log_params(self.config.all_params)
    
            mlflow.log_metric("rmse",rmse)
            mlflow.log_metric("mae",mae)
            mlflow.log_metric("r2",r2)
            
            if tracking_url_type_store!="file":
                mlflow.sklearn.log_model(model,"model",registered_model_name="ElasticNet Model")
            else:
                mlflow.sklearn.log_model(model,"model")
        

In [9]:
try:
    config=ConfiguationManager()
    model_evaluation_config=config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-09-14 16:40:34,085: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-14 16:40:34,290: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-14 16:40:34,298: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-14 16:40:34,335: INFO: common: created directory at: artifacts]
[2026-09-14 16:40:34,340: INFO: common: created directory at: artifacts/model_evaluation]
[2026-09-14 16:40:44,366: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/09/14 16:40:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/14 16:44:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\prfa9\AppData\Local\Temp\tmpattx2max\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 
Successfully registered model 'ElasticNet Model'.
2026/09/14 16:44:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet Model, version 1
Created version '1' of model 'ElasticNet Model'.


🏃 View run dapper-grub-215 at: https://dagshub.com/parastoof/data-science-project.mlflow/#/experiments/0/runs/92a031f8e3654328af52757e25e5619d
🧪 View experiment at: https://dagshub.com/parastoof/data-science-project.mlflow/#/experiments/0
